In [2]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    average_precision_score, classification_report, confusion_matrix, f1_score
)

RANDOM_STATE = 2137

In [3]:
df = pd.read_csv("data_for_classifier.csv")

In [4]:
df = df.drop(columns = "datetime")

print(df.columns[df.isna().any()].tolist())

['merchant_info.merchant_id', 'merchant_info.category', 'merchant_info.risk_level', 'merchant_info.avg_transaction', 'orig_count_as_orig_1h', 'orig_mean_amt_as_orig_1h', 'orig_unique_dest_1h', 'orig_count_as_orig_24h', 'orig_mean_amt_as_orig_24h', 'orig_unique_dest_24h', 'orig_count_as_orig_7d', 'orig_mean_amt_as_orig_7d', 'orig_unique_dest_7d', 'orig_usual_amt_as_orig', 'orig_time_since_last_tx_as_orig', 'orig_count_as_dest_1h', 'orig_mean_amt_as_dest_1h', 'orig_unique_orig_1h', 'orig_count_as_dest_24h', 'orig_mean_amt_as_dest_24h', 'orig_unique_orig_24h', 'orig_count_as_dest_7d', 'orig_mean_amt_as_dest_7d', 'orig_unique_orig_7d', 'orig_all_time_count_as_dest', 'orig_usual_amt_as_dest', 'orig_usual_count_per_hour_as_dest', 'orig_usual_volume_per_hour_as_dest', 'orig_time_since_last_tx_as_dest', 'dest_count_as_orig_1h', 'dest_mean_amt_as_orig_1h', 'dest_unique_dest_1h', 'dest_count_as_orig_24h', 'dest_mean_amt_as_orig_24h', 'dest_unique_dest_24h', 'dest_count_as_orig_7d', 'dest_mean_am

In [5]:
df = df.loc[:, ~df.columns.str.startswith("merchant_info.")]
df = df.fillna(0)

In [6]:
split_idx = int(len(df) * 0.9)
train = df.iloc[:split_idx]
test = df.iloc[split_idx:]

X_train = train.drop(columns = "is_fraud")
y_train = train["is_fraud"]

X_test = test.drop(columns = "is_fraud")
y_test = test["is_fraud"]

In [7]:
val_split_idx = int(len(train) * (8/9))

X_train_sub = X_train[:val_split_idx]
y_train_sub = y_train[:val_split_idx]

X_val = X_train[val_split_idx:]
y_val = y_train[val_split_idx:]

In [8]:
# Scale features and weight the rare positive class during fitting.
model = Pipeline([
    ("scaler", StandardScaler()),
    ("logistic_regression", LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=RANDOM_STATE,
    )),
])

# Choose the alert threshold on the validation slice only.
model.fit(X_train_sub, y_train_sub)
val_scores = model.predict_proba(X_val)[:, 1]
thresholds = np.unique(np.quantile(val_scores, np.linspace(0, 1, 501)))
val_f1 = [
    f1_score(y_val, val_scores >= threshold, zero_division=0)
    for threshold in thresholds
]
best_threshold = float(thresholds[int(np.argmax(val_f1))])
print(f"Validation-selected threshold: {best_threshold:.6f}")
print(f"Validation F1: {max(val_f1):.4f}")

# Refit on all pre-test rows and evaluate the held-out latest 10% once.
model.fit(X_train, y_train)
test_scores = model.predict_proba(X_test)[:, 1]
y_pred = (test_scores >= best_threshold).astype(int)
print(f"Test average precision: {average_precision_score(y_test, test_scores):.4f}")
print("Test classification report:")
print(classification_report(y_test, y_pred, zero_division=0))
print("Confusion matrix [[TN, FP], [FN, TP]]:")
print(confusion_matrix(y_test, y_pred, labels=[0, 1]))

Validation-selected threshold: 0.983965
Validation F1: 0.6845
Test average precision: 0.8231
Test classification report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    108833
           1       0.76      0.92      0.84       185

    accuracy                           1.00    109018
   macro avg       0.88      0.96      0.92    109018
weighted avg       1.00      1.00      1.00    109018

Confusion matrix [[TN, FP], [FN, TP]]:
[[108780     53]
 [    14    171]]
